# BKinD on CalMS21 (noise runs) — Colab

Same training as the Kaggle notebook (`bkind_calms21_kaggle.ipynb`): all the work is done by
`scripts/kaggle_bkind.py`, cloned fresh from GitHub each session, so change code there and push.
Colab has one GPU, so run **one** run per session. One run takes ~10–11 h on a T4 (faster on L4/A100).

**Setup (once):**
1. The frames are downloaded from the Kaggle dataset `sleekeagle/calms21-train` with kagglehub
   (the same data the Kaggle notebook uses). Google Drive holds only the run outputs.
2. *Runtime → Change runtime type →* a **GPU** (T4 or better).
3. If the GitHub repo is private: Secrets panel (key icon), add `GITHUB_TOKEN` with read access.

Checkpoints, logs and samples are saved to `MyDrive/noisykeypoints/runs/<name>/` after every epoch.
**If Colab disconnects:** reconnect and *Run all* again with the same `RUNS`; training continues from
the last finished epoch (at most one epoch, ~45 min, is lost).


In [ ]:
# ---- Settings ----
# One run per session. Sensor noise at SSIM 0.9 already mostly collapsed, so look for the edge
# with milder noise (explicit photons; read = 0.6 electrons).
def pg_ssim(ssim, photons, seed):
    return dict(name=f'CalMS21_pg_ssim{ssim}_s{seed}', seed=seed,
                overrides=dict(noise=dict(type='poisson_gaussian', photons=photons, read=0.6 / photons, seed=0)))
RUNS = [pg_ssim(99, 10000.0, 2)]    # SSIM 0.990 (44.2 dB), seed 2: does 'works' repeat at the edge?
# Queue (one per session):
#   pg_ssim(98, 5000.0, 2)          # SSIM 0.980, seed 2
RUNS_PER_GPU = 1        # 2 on one GPU was no faster (0.41 s/step each vs 0.20 alone)
# Queue (one per session):
#   (ll_L2_frozen_s1 moved to Kaggle)
# Done: pg_ssim99 -> works (~6 kpts on mice, near clean); pg_ssim98 -> 2 keypoints track; pg_ssim95 -> mostly collapsed; ll_L2, sp_L2, blur_L2 -> collapsed; h264_L2 -> works (weaker); blur_L2_frozen, sp_L2_frozen -> work.
REPO_URL = 'https://github.com/sleekEagle/noisykeypoints.git'
MAX_HOURS = 11.5        # stop cleanly (between epochs) before this many hours into the session

import os, sys, time, subprocess
T0 = time.time()
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/noisykeypoints'

# The CalMS21 frames, from the Kaggle dataset (cached for the rest of this Colab session).
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'kagglehub'], check=True)
import kagglehub
DATA_IN = kagglehub.dataset_download('sleekeagle/calms21-train')
print('CalMS21 download:', DATA_IN)

# Get the code (uses the GITHUB_TOKEN secret if the repo is private) and load the helper.
CODE = '/content/noisykeypoints'
url = REPO_URL
try:
    from google.colab import userdata
    url = REPO_URL.replace('https://', f'https://{userdata.get("GITHUB_TOKEN")}@')
except Exception:
    print('no GITHUB_TOKEN secret; cloning as a public repo')
if os.path.exists(CODE):
    subprocess.run(['git', '-C', CODE, 'pull', '-q'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', url, CODE], check=True)
sys.path.insert(0, f'{CODE}/scripts')
import kaggle_bkind


In [ ]:
runs = kaggle_bkind.run('calms21', RUNS, max_hours=MAX_HOURS, t0=T0,
                        inp=DATA_IN, data='/content/data', out=f'{DRIVE}/runs',
                        runs_per_gpu=RUNS_PER_GPU)
